# 01 Data Generation or Ingestion

This notebook generates synthetic Type 1 diabetes patient and glucose data for the analytics project.

Outputs:
- data/raw/patients.csv
- data/raw/glucose_readings.csv
- data/raw/insulin_events.csv

In [1]:
# Import necessary libraries for data manipulation, random number generation, file paths, and date handling
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

In [2]:
# Random number generator with fixed seed for reproducible results
rng = np.random.default_rng(seed=42)

# Define project root directory and create raw data path
project_root = Path.cwd().parent
raw_data_path = project_root / "data" / "raw"
raw_data_path.mkdir(parents=True, exist_ok=True)

print("Raw data path:", raw_data_path)

Raw data path: c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw


In [3]:
# Configuration parameters for data generation
n_patients = 10 
start_date = pd.Timestamp("2026-01-01") 
num_days = 7 
glucose_freq_minutes = 30 

In [4]:
# Lists of possible values for categorical patient attributes
sexes = ["Female", "Male"]
age_groups = ["18-24", "25-34", "35-44", "45-54", "55-64"]
insulin_methods = ["Pump", "Multiple Daily Injections"]
cgm_devices = ["Dexcom", "Libre", "Medtronic"]

# Generate synthetic patient data as a DataFrame
patients = pd.DataFrame({
    "patient_id": range(1, n_patients + 1),  # Unique identifier for each patient
    "sex": rng.choice(sexes, n_patients),  # Randomly assigned sex
    "age_group": rng.choice(age_groups, n_patients),  # Randomly assigned age group
    "diabetes_duration_years": rng.integers(1, 31, n_patients),  # Years since diabetes diagnosis (1-30)
    "insulin_delivery_method": rng.choice(insulin_methods, n_patients),  # Insulin delivery method
    "cgm_device_type": rng.choice(cgm_devices, n_patients)  # Type of continuous glucose monitor
})

patients.head()

,patient_id,sex,age_group,diabetes_duration_years,insulin_delivery_method,cgm_device_type
0,1,Female,35-44,16,Pump,Dexcom
1,2,Male,55-64,12,Pump,Medtronic
2,3,Male,45-54,6,Pump,Medtronic
3,4,Female,45-54,28,Multiple Daily Injections,Libre
4,5,Female,45-54,24,Multiple Daily Injections,Dexcom


In [5]:
# Generate timestamps for glucose measurements across the specified period
timestamps = pd.date_range(
    start=start_date,
    periods=(24 * 60 // glucose_freq_minutes) * num_days,  # Total measurements per day times days
    freq=f"{glucose_freq_minutes}min"  # Frequency string for date range
)

len(timestamps), timestamps[:5]

(336,
 DatetimeIndex(['2026-01-01 00:00:00', '2026-01-01 00:30:00',
                '2026-01-01 01:00:00', '2026-01-01 01:30:00',
                '2026-01-01 02:00:00'],
               dtype='datetime64[us]', freq='30min'))

In [7]:
# Generate synthetic glucose readings for each patient
glucose_rows = []
measurement_id = 1 

for patient_id in patients["patient_id"]:
    # Baseline glucose level and variability for each patient
    baseline = rng.normal(145, 20)
    variability = rng.uniform(15, 40)

    for ts in timestamps:
        hour = ts.hour

        # Simulate meal effects on glucose levels based on time of day
        if 6 <= hour <= 9:  # Breakfast time
            meal_effect = rng.normal(15, 10)  # Increase due to breakfast
        elif 12 <= hour <= 14:  # Lunch time
            meal_effect = rng.normal(20, 12)  # Increase due to lunch
        elif 18 <= hour <= 21:  # Dinner time
            meal_effect = rng.normal(25, 15)  # Increase due to dinner
        elif 0 <= hour <= 5:  # Overnight
            meal_effect = rng.normal(-10, 8)  # Slight decrease overnight
        else:
            meal_effect = rng.normal(0, 8)  # Baseline effect for other times

        # Calculate glucose value with baseline, meal effect, and random noise
        glucose_value = baseline + meal_effect + rng.normal(0, variability)
        glucose_value = max(40, min(400, round(glucose_value, 2)))

        glucose_rows.append({
            "measurement_id": measurement_id,
            "patient_id": patient_id,
            "measurement_timestamp": ts,
            "glucose_mg_dl": glucose_value
        })

        measurement_id += 1

glucose = pd.DataFrame(glucose_rows)
glucose.head()

,measurement_id,patient_id,measurement_timestamp,glucose_mg_dl
0,1,1,2026-01-01 00:00:00,118.46
1,2,1,2026-01-01 00:30:00,91.75
2,3,1,2026-01-01 01:00:00,87.90
3,4,1,2026-01-01 01:30:00,106.59
4,5,1,2026-01-01 02:00:00,109.15


In [8]:
# Generate synthetic insulin administration events for each patient
insulin_rows = []
insulin_event_id = 1

for _, patient in patients.iterrows():
    patient_id = patient["patient_id"]
    patient_method = patient["insulin_delivery_method"]

    # Assign a consistent event-level delivery method per patient
    if patient_method == "Pump":
        patient_delivery_method = "Pump"
    else:
        patient_delivery_method = str(rng.choice(["Pen", "Syringe"]))

    for day in pd.date_range(start=start_date, periods=num_days, freq="D"):
        if patient_method == "Pump":
            # Pump users: bolus events only in this simplified simulation
            daily_events = [
                {
                    "administration_timestamp": day + pd.Timedelta(hours=7, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(3, 8)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                },
                {
                    "administration_timestamp": day + pd.Timedelta(hours=12, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(4, 10)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                },
                {
                    "administration_timestamp": day + pd.Timedelta(hours=18, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(5, 12)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                }
            ]

        else:
            basal_pattern = rng.choice(["morning", "evening", "split"], p=[0.4, 0.3, 0.3])

            if basal_pattern == "morning":
                daily_events = [
                    {
                        "administration_timestamp": day + pd.Timedelta(hours=8, minutes=int(rng.integers(-30, 31))),
                        "dose_units": round(float(rng.uniform(12, 28)), 2),
                        "delivery_method": patient_delivery_method,
                        "insulin_type": "Basal"
                    }
                ]
            elif basal_pattern == "evening":
                daily_events = [
                    {
                        "administration_timestamp": day + pd.Timedelta(hours=20, minutes=int(rng.integers(-30, 31))),
                        "dose_units": round(float(rng.uniform(12, 28)), 2),
                        "delivery_method": patient_delivery_method,
                        "insulin_type": "Basal"
                    }
                ]
            else:
                total_basal = rng.uniform(12, 28)
                morning_fraction = rng.uniform(0.4, 0.7)
                morning_dose = round(float(total_basal * morning_fraction), 2)
                evening_dose = round(float(total_basal - morning_dose), 2)

                daily_events = [
                    {
                        "administration_timestamp": day + pd.Timedelta(hours=8, minutes=int(rng.integers(-30, 31))),
                        "dose_units": morning_dose,
                        "delivery_method": patient_delivery_method,
                        "insulin_type": "Basal"
                    },
                    {
                        "administration_timestamp": day + pd.Timedelta(hours=20, minutes=int(rng.integers(-30, 31))),
                        "dose_units": evening_dose,
                        "delivery_method": patient_delivery_method,
                        "insulin_type": "Basal"
                    }
                ]

            bolus_events = [
                {
                    "administration_timestamp": day + pd.Timedelta(hours=7, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(3, 8)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                },
                {
                    "administration_timestamp": day + pd.Timedelta(hours=12, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(4, 10)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                },
                {
                    "administration_timestamp": day + pd.Timedelta(hours=18, minutes=int(rng.integers(-20, 21))),
                    "dose_units": round(float(rng.uniform(5, 12)), 2),
                    "delivery_method": patient_delivery_method,
                    "insulin_type": "Bolus"
                }
            ]

            daily_events.extend(bolus_events)

        for event in daily_events:
            insulin_rows.append({
                "insulin_event_id": insulin_event_id,
                "patient_id": patient_id,
                "administration_timestamp": event["administration_timestamp"],
                "dose_units": event["dose_units"],
                "delivery_method": event["delivery_method"],
                "insulin_type": event["insulin_type"]
            })
            insulin_event_id += 1

insulin = pd.DataFrame(insulin_rows).sort_values(
    ["patient_id", "administration_timestamp", "insulin_event_id"]
).reset_index(drop=True)

insulin.head()

,insulin_event_id,patient_id,administration_timestamp,dose_units,delivery_method,insulin_type
0,1,1,2026-01-01 07:00:00,7.39,Pump,Bolus
1,2,1,2026-01-01 11:51:00,4.33,Pump,Bolus
2,3,1,2026-01-01 17:54:00,8.76,Pump,Bolus
3,4,1,2026-01-02 06:52:00,4.70,Pump,Bolus
4,5,1,2026-01-02 12:10:00,5.59,Pump,Bolus


In [9]:
# Display data summary and previews
print("Patients shape:", patients.shape)
print("Glucose shape:", glucose.shape) 
print("Insulin shape:", insulin.shape)  

print("\nPatients preview:")
display(patients.head())

print("\nGlucose preview:")
display(glucose.head())

print("\nInsulin preview:")
display(insulin.head())

print("\nGlucose summary:")
display(glucose["glucose_mg_dl"].describe())

print("\nInsulin summary:")
display(insulin["dose_units"].describe())

Patients shape: (10, 6)
Glucose shape: (3360, 4)
Insulin shape: (253, 6)

Patients preview:


,patient_id,sex,age_group,diabetes_duration_years,insulin_delivery_method,cgm_device_type
0,1,Female,35-44,16,Pump,Dexcom
1,2,Male,55-64,12,Pump,Medtronic
2,3,Male,45-54,6,Pump,Medtronic
3,4,Female,45-54,28,Multiple Daily Injections,Libre
4,5,Female,45-54,24,Multiple Daily Injections,Dexcom



Glucose preview:


,measurement_id,patient_id,measurement_timestamp,glucose_mg_dl
0,1,1,2026-01-01 00:00:00,118.46
1,2,1,2026-01-01 00:30:00,91.75
2,3,1,2026-01-01 01:00:00,87.90
3,4,1,2026-01-01 01:30:00,106.59
4,5,1,2026-01-01 02:00:00,109.15



Insulin preview:


,insulin_event_id,patient_id,administration_timestamp,dose_units,delivery_method,insulin_type
0,1,1,2026-01-01 07:00:00,7.39,Pump,Bolus
1,2,1,2026-01-01 11:51:00,4.33,Pump,Bolus
2,3,1,2026-01-01 17:54:00,8.76,Pump,Bolus
3,4,1,2026-01-02 06:52:00,4.70,Pump,Bolus
4,5,1,2026-01-02 12:10:00,5.59,Pump,Bolus



Glucose summary:


count    3360.000000
mean      145.151149
std        42.418202
min        40.000000
25%       114.122500
50%       143.375000
75%       174.747500
max       290.080000
Name: glucose_mg_dl, dtype: float64


Insulin summary:


count    253.000000
mean       8.559012
std        4.559018
min        3.010000
25%        5.690000
50%        7.630000
75%        9.710000
max       27.750000
Name: dose_units, dtype: float64

In [10]:
# Data quality checks: check for null values and duplicates
print("Nulls in patients:")
print(patients.isnull().sum())

print("\nNulls in glucose:")
print(glucose.isnull().sum())

print("\nNulls in insulin:")
print(insulin.isnull().sum())

print("\nDuplicate patient IDs:", patients["patient_id"].duplicated().sum())
print("Duplicate measurement IDs:", glucose["measurement_id"].duplicated().sum())
print("Duplicate insulin event IDs:", insulin["insulin_event_id"].duplicated().sum())

Nulls in patients:
patient_id                 0
sex                        0
age_group                  0
diabetes_duration_years    0
insulin_delivery_method    0
cgm_device_type            0
dtype: int64

Nulls in glucose:
measurement_id           0
patient_id               0
measurement_timestamp    0
glucose_mg_dl            0
dtype: int64

Nulls in insulin:
insulin_event_id            0
patient_id                  0
administration_timestamp    0
dose_units                  0
delivery_method             0
insulin_type                0
dtype: int64

Duplicate patient IDs: 0
Duplicate measurement IDs: 0
Duplicate insulin event IDs: 0


In [11]:
# Save generated data to CSV files.
patients.to_csv(raw_data_path / "patients.csv", index=False)
glucose.to_csv(raw_data_path / "glucose_readings.csv", index=False)
insulin.to_csv(raw_data_path / "insulin_events.csv", index=False)

print("Saved:")
print(raw_data_path / "patients.csv") 
print(raw_data_path / "glucose_readings.csv")
print(raw_data_path / "insulin_events.csv")

Saved:
c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw\patients.csv
c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw\glucose_readings.csv
c:\Users\Joshua\Documents\type1-diabetes-analytics\data\raw\insulin_events.csv
